# Knowledge Graphs and Semantic Technologies -- RDF tutorial

In this tutorial we'll learn the basics of interacting with RDF graphs with Python. We'll be using rdflib for this, a widely used Ptyhon library for RDF (all documentation can be found [here](https://rdflib.readthedocs.io/en/stable/index.html))

## Imports
These are the main classes and types we'll be using from rdflib

In [1]:
import sys

from rdflib import Graph, ConjunctiveGraph, Literal, BNode, Namespace, RDF, URIRef
from rdflib.namespace import DC, FOAF

import pprint


## Loading data remotely and from files

rdflib accepts importing RDF data from a variety of sources, either locally from a file (including an extensive support of serializations), or remotely via a URI (this is a great way of checking practically if URIs return RDF according to the 3rd Linked Data principle).

A Graph object is always required to load triples.
**Note**: to load quads, and hence supporting named graphs, you'll need to use an instance of ConjunctiveGraph instead

**Exercise 1** 

For each step, use a different cell: 
1. create two graphs using rdflib:
    - and load one with triples from the site https://csarven.ca/ and/or http://www.w3.org/People/Berners-Lee/card 
    - load one with triples from ./data/ingredients.rdf. 

In [2]:
 #TIP: look at the documentation of the rdflib library for how to LOAD and PARSE a graph - https://rdflib.readthedocs.io/en/stable/gettingstarted.html
loaded_graph1 = Graph()
loaded_graph1.parse("http://www.w3.org/People/Berners-Lee/card")
print(loaded_graph1)
# loaded_graph2 = Graph()
# loaded_graph2.parse("https://csarven.ca")
# print(len(loaded_graph1))

local_graph = Graph()
local_graph.parse("data/ingredients.rdf")
print(local_graph)

[a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'Memory']].
[a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'Memory']].


## Serialising and saving RDF graphs

There are different formats for storing RDF triples. Semantically, these mean the same, they differ only in their syntax. 


Use the function Graph.serialize(format). 

**Exercise 2**

1. serialise one of the graphs to the .ttl, .xml and .nt format, and print the first n lines to compare the syntax
1. save your graph in the turtle format to the ./data/ folder

In [3]:
# serialize the chosen graph into different formats and save
loaded_graph1.serialize(destination="data/berners_lee.ttl", format="turtle")
loaded_graph1.serialize(destination="data/berners_lee.xml", format="xml")
loaded_graph1.serialize(destination="data/berners_lee.nt", format="nt", encoding="utf-8")


<Graph identifier=Nfc9814a356fa4cd9a3d0ab9897af4e51 (<class 'rdflib.graph.Graph'>)>

##  Merging graphs

Merging graphs can be done via sequential parsings or by the overloaded operator +

**Note:** Set-theoretic graph semantics apply

The Food knowledge graph FoodKG contains a graph of statements about ingredients, as well as a graph with statements about recipes. 

**Exercise 3**: 

1. load ./data/ingredients.rdf and ./data/ghostbusters.ttl into a single graph, either by sequential parsing or using the operator +.

2. count the number of statements in each graph, and the intersection of the two graphs. 

3. check whether the combined graph is connected (using graph.connected()) 

4. load ./data/ingredients.rdf and ./data/recipes.rdf into a single graph, either by sequential parsing or using the operator +. 

5. count the number of statements in each graph, and the intersection of the two graphs. 

6. check whether the combined graph is connected (using graph.connected()). Explain the result with respect to point 3! 

In [4]:
# Look at rdflib documentation - Navigating Graphs

# 1. Load the first graph, then through the + operator add the second graph
load_ingredients_graph = Graph()
load_ingredients_graph.parse("data/ingredients.rdf")

load_ghost_graph = Graph()
load_ghost_graph.parse("data/ghostbusters.ttl")

merged_graph = load_ingredients_graph + load_ghost_graph
print(f'Exercise 3.1\nMerged graph has {len(merged_graph)} triples\n')

# 2. Count the number of statements in each graph and the intersection = triples that appear identically in both graphs
print("Exercise 3.2")
print("Triples in the ingredients graph:", len(load_ingredients_graph))
print("Triples in the ghostbuster graph:", len(load_ghost_graph))

intersection_graph = set(load_ingredients_graph) & set(load_ghost_graph)
print("Triples in the intersection of both graphs:", len(intersection_graph), "\n")

# 3. Check if the combined graph is connected
is_graph_connected = merged_graph.connected()
print(f'Exercise 3.3\nis the combined graph connected: {is_graph_connected}\n')

# 4. The first graph is already loaded. Load the second graph then merge them through the + operator
load_recipe_graph = Graph()
load_recipe_graph.parse("data/recipes.rdf")

new_merged_graph = load_ingredients_graph + load_recipe_graph
print(f'Exercise 3.4\nMerged graph has {len(new_merged_graph)} triples\n')

# 5. Count the number of statements and intersection = triples that appear identically in both graphs
print("Exercise 3.5")
print("Triples in the ingredients graph:", len(load_ingredients_graph))
print("Triples in the recipe graph:", len(load_recipe_graph))

intersection_new_graph = set(load_ingredients_graph) & set(load_recipe_graph)
print("Triples in the intersection of both graphs:", len(intersection_new_graph), "\n")

# 6. Check if the combined graph is connected
is_new_graph_connected = new_merged_graph.connected()
print(f'Exercise 3.6\nis the combined graph connected: {is_new_graph_connected}\n')

"""
The merged graph is not connected, because the two graphs do not share enough common nodes (subjects or objects) to form a connected graph.
Connectivity depends on shared nodes, not on the number of overlapping triples. Even though there are no connecting triples in the merged graph
in assignment 3, there is still no connecting graph in this assignment either as the graphs do not fully overlap.
"""

Exercise 3.1
Merged graph has 53174 triples

Exercise 3.2
Triples in the ingredients graph: 837
Triples in the ghostbuster graph: 52337
Triples in the intersection of both graphs: 0 

Exercise 3.3
is the combined graph connected: False

Exercise 3.4
Merged graph has 1299 triples

Exercise 3.5
Triples in the ingredients graph: 837
Triples in the recipe graph: 480
Triples in the intersection of both graphs: 18 

Exercise 3.6
is the combined graph connected: False



'\nThe merged graph is not connected, because the two graphs do not share enough common nodes (subjects or objects) to form a connected graph.\nConnectivity depends on shared nodes, not on the number of overlapping triples. Even though there are no connecting triples in the merged graph\nin assignment 3, there is still no connecting graph in this assignment either as the graphs do not fully overlap.\n'

## Namespaces 

Remind yourself what namespaces are. 

In RDFLib, the namespace module defines many common namespaces such as RDF, RDFS, OWL, FOAF, SKOS, etc., but you can also easily add URIs within a different namespace:


In [5]:
TEACH = Namespace("http://linkedscience.org/teach/ns#")
TEACH.Teacher

rdflib.term.URIRef('http://linkedscience.org/teach/ns#Teacher')

Check out the specification to see which other terms are used within the TEACH namespace. http://linkedscience.org/teach/ns/#sec-specification. 
You can use a NamespaceManager to bind a prefix to a namespace: 

In [6]:
g = Graph()
g.namespace_manager.bind('TEACH', URIRef('http://linkedscience.org/teach/ns#'))
TEACH.Teacher.n3(g.namespace_manager)

'TEACH:Teacher'

In [7]:
KRW = Namespace("http://krw.vu.nl/data#")

# Creating individuals within your namespace
KRW.Teacher
KRW.Student

g.namespace_manager.bind('krw', KRW)

print(KRW.Teacher.n3(g.namespace_manager))
print(KRW.Student.n3(g.namespace_manager))

krw:Teacher
krw:Student


**Exercise 4:**
1. create your own namespace (can be made up)

In [8]:
# Code here
namespace_graph = Graph()
KIM = Namespace("https://mynameis/kim#")

namespace_graph.namespace_manager.bind("kim", KIM)

KIM.Student.n3(namespace_graph.namespace_manager)

'kim:Student'


## Creating RDF triples

Triples are added to the graph with the function Graph.add()

The parameter is a triple given in a Python **tuple** (subject, predicate, object)

Notice the namespace convenience syntax!

**Exercise 5:** 

1. create a new graph and add triples (~10) within your made-up namespace using Graph.add(). These triples can be about anything, for instance ingredients or recipes. Make sure they include the predicates RDF.type, RDFS.label and RDFS.subClassOf

2. open yourRDF.ttl, and write your triples out by hand in a syntax of your choice (turtle is recommended, notice the file extension!). Load the triples here with rdflib. 

In [9]:
from rdflib import RDFS

# Create a graph
my_namespace_graph = Graph()

# Example namespace
KIM = Namespace("https://mynameis/kim#")
my_namespace_graph.namespace_manager.bind("kim", KIM)

# Add triples using store's add method.
my_namespace_graph.add((KIM.Person, RDF.type, RDFS.Class))
my_namespace_graph.add((KIM.Name, RDF.type, RDFS.Class))
my_namespace_graph.add((KIM.Name, RDFS.label, Literal("Name")))
my_namespace_graph.add((KIM.Student, RDFS.subClassOf, KIM.Person))
my_namespace_graph.add((KIM.Age, RDF.type, RDF.Property))
my_namespace_graph.add((KIM.Age, RDFS.label, Literal("Age")))
my_namespace_graph.add((KIM.City, RDF.type, RDFS.Class))
my_namespace_graph.add((KIM.Amsterdam, RDF.Type, KIM.City))
my_namespace_graph.add((KIM.Amsterdam, RDFS.label, Literal("Amsterdam")))
my_namespace_graph.add((KIM.University, RDF.type, RDFS.Class))
my_namespace_graph.add((KIM.VU, RDFS.subClassOf, KIM.University))

# Save the graph to destination in ttl format - myRDF.ttl (look at RDFLib documentation - Loading and saving RDF)
my_namespace_graph.serialize(destination="data/kim.ttl", format="turtle")

# Load the saved graph and print it in ttl format
load_my_namespace_graph = Graph()
load_my_namespace_graph.parse("data/kim.ttl")

print(load_my_namespace_graph.serialize(format="turtle"))

@prefix kim: <https://mynameis/kim#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

kim:City a rdfs:Class .

kim:Name a rdfs:Class ;
    rdfs:label "Name" .

kim:Person a rdfs:Class .

kim:University a rdfs:Class ;
    rdfs:subClassOf kim:University .

kim:Age a rdf:Property ;
    rdfs:label "Age" .

kim:Amsterdam rdfs:label "Amsterdam" ;
    rdfs:subClassOf kim:City .

kim:Student rdfs:subClassOf kim:Person .




## Navigating graphs

rdflib uses iterators to navigate Graphs. The methods for navigating subjects, predicates and objects are Graph.subjects, Graph.predicates, Graph.objects

**Exercise 6:**

1. print all the triples in yourRDF.ttl
2. print all subjects in yourRDF.ttl
3. print all predicates in yourRDF.ttl
4. print all objects in yourRDF.ttl


In [10]:
# TIP: you have to loop in the graph
load_my_namespace_graph = Graph()
load_my_namespace_graph.parse("data/kim.ttl")

# 1. Print all triples
print(f'Exercise 6.1\nPrinting all tripled')
for sub, pred, obj in load_my_namespace_graph:
    print(sub, pred, obj)

# 2. Print all subjects
print(f'\nExercise 6.2\nPrinting all subjects')
for sub in load_my_namespace_graph:
    print(sub)

# 3. Print all predicates
print(f'\nExercise 6.3\nPrinting all predicates')
for pred in load_my_namespace_graph:
    print(pred)

# 4. Print all objects
print(f'\nExercise 6.4\nPrinting all objects')
for obj in load_my_namespace_graph:
    print(obj)

Exercise 6.1
Printing all tripled
https://mynameis/kim#University http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Class
https://mynameis/kim#Age http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
https://mynameis/kim#Amsterdam http://www.w3.org/2000/01/rdf-schema#subClassOf https://mynameis/kim#City
https://mynameis/kim#University http://www.w3.org/2000/01/rdf-schema#subClassOf https://mynameis/kim#University
https://mynameis/kim#Name http://www.w3.org/2000/01/rdf-schema#label Name
https://mynameis/kim#Amsterdam http://www.w3.org/2000/01/rdf-schema#label Amsterdam
https://mynameis/kim#City http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Class
https://mynameis/kim#Name http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Class
https://mynameis/kim#Age http://www.w3.org/2000/01/rdf-schema#label Age
https://mynameis/kim#Student http://w

We can also filter the subjects, predicates and objects we want to retrieve, and match their values like in a database "join" operation


**Exercise 7:**

1. print all subject types in yourRDF.ttl
2. print all subject labels yourRDF.ttl

In [11]:
# 1. Print all subject types
print(f'\nExercise 7.1\nPrinting all subject types')
for subType in load_my_namespace_graph.subjects(RDF.type):
    print(subType)

# 2. Print all subject labels
print(f'\nExercise 7.1\nPrinting all subject labels')
for subLabel in load_my_namespace_graph.subjects(RDFS.label):
    print(subLabel)


Exercise 7.1
Printing all subject types
https://mynameis/kim#City
https://mynameis/kim#Name
https://mynameis/kim#Person
https://mynameis/kim#University
https://mynameis/kim#Age

Exercise 7.1
Printing all subject labels
https://mynameis/kim#Name
https://mynameis/kim#Age
https://mynameis/kim#Amsterdam


### Basic triple matching (almost querying!)

We use method Graph.triples and a Python tuple that acts as a mask for specifying our criteria

**Exercise 8:**

1. check whether a triple is in your graph -> print true or false
2. print all triples related to a certain subject in your graph
3. print all triples related to a certain object in your graph

In [12]:
load_my_namespace_graph = Graph()
load_my_namespace_graph.parse("data/kim.ttl")

# 1. Check whether a triple is in the graph. For example, check KIM:Kim RDF.type KIM:Person
print(f'\nExercise 8.1: Check if a certain triple exists:')
check = load_my_namespace_graph.triples((KIM.Kim, RDF.type, KIM.Person))
print(any(check))

# 2. Print all triples related to a certain subject
print("\nExercise 8.3: All triples related to subject KIM:City")
for sub, pred, obj in load_my_namespace_graph.triples((KIM.City, None, None)):
    print(sub, pred, obj)

# 3. Print all triples related to a certain object
print("\nExercise 8.3: All triples related to object KIM:Person")
for sub, pred, obj in load_my_namespace_graph.triples((None, None, KIM.Person)):
    print(sub, pred, obj)


Exercise 8.1: Check if a certain triple exists:
False

Exercise 8.3: All triples related to subject KIM:City
https://mynameis/kim#City http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Class

Exercise 8.3: All triples related to object KIM:Person
https://mynameis/kim#Student http://www.w3.org/2000/01/rdf-schema#subClassOf https://mynameis/kim#Person


## Restaurant Exercise - Part 1

You are a chef in a restaurant, and you need to serve someone that is gluten intolerant. 

1. load the ./data/recipes.rdf and ./data/ingredients.rdf datasets in one graph
2. query your graph (as we did in previous exercises) to retrieve all recipes without gluten
3. query your graph for all recipes that you can make for your gluten intolerant guest. 
4. the guest asks you whether there are more options. Can you find the recipes for which an ingredient with gluten can be replaced, solely using pattern matching? (Hint: you need to write multiple of these pattern matching queries, and check the predicate __substitutesFor__) 
5. another guest is allergic to pecan nuts, which recipes could you serve them (including those for which pecan nuts can be replaced) 

**Note that this is a bit tedious: later on, we will be querying more complicated patterns with SPARQL!**

In [13]:
# 1. Reload the graphs and merge
load_ingredients_graph = Graph()
load_ingredients_graph.parse("data/ingredients.rdf")

load_recipe_graph = Graph()
load_recipe_graph.parse("data/recipes.rdf")

restaurant_graph = load_ingredients_graph + load_recipe_graph

# 1.1 Get some information out of the graphs
# a) Find the namespace for the representation of Recipes
print(f"\nFind the namespace for Recipes")
for _, _, obj in restaurant_graph.triples((None, RDF.type, None)):
    if "Recipe" in str(obj):
        print(obj)
        break

# b)  Find the namespace for the representation of Gluten
print(f"\nFind the namespace for Gluten")
for _, pred, _ in restaurant_graph:
    if "gluten" in str(pred).lower():
        print(pred)
        break

# c) Find the property linking recipes to ingredients
print(f"\nFind the properties for linking recipes to ingredients")
for _, pred, obj in restaurant_graph:
    if "ingredient" in str(pred).lower():
        print(obj)
        break


Find the namespace for Recipes
http://purl.org/heals/food/Recipe

Find the namespace for Gluten
http://purl.org/heals/food/hasGluten

Find the properties for linking recipes to ingredients
http://purl.org/heals/ingredient/GreenOnion


In [14]:
# d) Define the namespace
FOOD = Namespace("http://purl.org/heals/food/")
INGREDIENT = Namespace("http://purl.org/heals/ingredient")

In [15]:
# 2. Query the graph to retrieve all recipes without gluten, together with
# 3. Query which recipes are able to be cooked with the ingredients available that are gluten-free
print("\nRestaurant Exercise, exercise 2: Gluten free recipes")
gluten_free_recipe_list = []
gluten_recipe_list = []
gluten_ingredient_list = []
gluten_ingredients = restaurant_graph.triples((None, FOOD.hasGluten, Literal(True)))

# Get all gluten ingredients
for sub, _, _ in gluten_ingredients:
    for subjects, _, _ in restaurant_graph.triples((None, FOOD.hasIngredient, sub)):
        gluten_ingredient_list.append(subjects)

# Check those gluten ingredients against the recipes to find the ones that are gluten-free
for sub, _, _ in restaurant_graph.triples((None, RDF.type, FOOD.Recipe)):
    if sub not in gluten_ingredient_list:
        gluten_free_recipe_list.append(sub)
        print(sub)
    else:
        gluten_recipe_list.append(sub)


Restaurant Exercise, exercise 2: Gluten free recipes
http://purl.org/heals/ingredient/BeefNilaga
http://purl.org/heals/ingredient/GrilledChickenKabob
http://purl.org/heals/ingredient/BeefStew
http://purl.org/heals/ingredient/SmotheredChickenBreast
http://purl.org/heals/ingredient/FlourlessCoconutAndAlmondCake
http://purl.org/heals/ingredient/BakedChickenTender
http://purl.org/heals/ingredient/CornedBeefHash
http://purl.org/heals/ingredient/BananaBlueberryAlmondFlourMuffin
http://purl.org/heals/ingredient/GlutenFreeCoconutCake
http://purl.org/heals/ingredient/SaucyShepherdPie
http://purl.org/heals/ingredient/PotRoastWithVegetables
http://purl.org/heals/ingredient/BraisedBalsamicChicken


In [16]:
# 4. Find substitutions for gluten
# a) Find the namespace for the predicate
print(f"\nExercise 4: Find substitutions for gluten. \nFind the namespace for the predicate substitution")
for _, pred, _ in restaurant_graph:
    if "substitute" in str(pred).lower():
        print(pred)
        break

gluten_ingredients = restaurant_graph.triples((None, FOOD.hasGluten, Literal(True)))

# Find substitution in the gluten_ingredients list
gluten_ingredients_substitution_list = []
for sub,_,_ in gluten_ingredients:
    for subjects, _, _ in restaurant_graph.triples( (None, FOOD.substitutesFor, sub)):
        gluten_ingredients_substitution_list.append(sub)

# Use the previous found gluten_recipe list to check which gluten recipes have ingredients that can be replaced
print("\nThese recipes have a gluten ingredient that can be replaced:")
for subj, _, _ in restaurant_graph.triples((None, RDF.type, FOOD.Recipe)):
    if subj in gluten_recipe_list:
        print(subj)



Exercise 4: Find substitutions for gluten. 
Find the namespace for the predicate substitution
http://purl.org/heals/food/substitutesFor

These recipes have a gluten ingredient that can be replaced:
http://purl.org/heals/ingredient/KamutPancake
http://purl.org/heals/ingredient/GoldenKamutBread
http://purl.org/heals/ingredient/KamutMuffin
http://purl.org/heals/ingredient/WhiteBread
http://purl.org/heals/ingredient/Brownies
http://purl.org/heals/ingredient/AlmondBiscotti
http://purl.org/heals/ingredient/ChickenSalad
http://purl.org/heals/ingredient/BananaBread
http://purl.org/heals/ingredient/ThaiChicken
http://purl.org/heals/ingredient/WholeGrainBananaPancake


In [17]:
# 5. Filter out pecan nuts
print(f'\nExercise 5: Pecan nuts allergy')
pecan_list = []
pecan_food = restaurant_graph.triples((None, None, Literal("pecan")))

for sub, pred, obj in pecan_food:
    for subject, _, _ in restaurant_graph.triples((None, FOOD.hasIngredient, sub)):
        pecan_list.append(subject)

print("Pecan-Free recipes")
for subject, _, _ in restaurant_graph.triples((None, RDF.type, FOOD.Recipe)):
     if subject not in pecan_list:
         print(subject)

# Find substitution in the pecan_list
pecan_substitution_list = []
for sub, _, _ in pecan_food:
    for subjects, _, _ in restaurant_graph.triples( (None, FOOD.substitutesFor, sub)):
       pecan_substitution_list.append(sub)

print("\nThese recipes have a pecan that can be replaced:")
for subj, _, _ in restaurant_graph.triples((None, RDF.type, FOOD.Recipe)):
    if subj in pecan_list:
        print(subj)




Exercise 5: Pecan nuts allergy
Pecan-Free recipes
http://purl.org/heals/ingredient/BeefNilaga
http://purl.org/heals/ingredient/GrilledChickenKabob
http://purl.org/heals/ingredient/BeefStew
http://purl.org/heals/ingredient/KamutPancake
http://purl.org/heals/ingredient/GoldenKamutBread
http://purl.org/heals/ingredient/KamutMuffin
http://purl.org/heals/ingredient/WhiteBread
http://purl.org/heals/ingredient/SmotheredChickenBreast
http://purl.org/heals/ingredient/FlourlessCoconutAndAlmondCake
http://purl.org/heals/ingredient/Brownies
http://purl.org/heals/ingredient/AlmondBiscotti
http://purl.org/heals/ingredient/BakedChickenTender
http://purl.org/heals/ingredient/CornedBeefHash
http://purl.org/heals/ingredient/BananaBread
http://purl.org/heals/ingredient/ThaiChicken
http://purl.org/heals/ingredient/BananaBlueberryAlmondFlourMuffin
http://purl.org/heals/ingredient/GlutenFreeCoconutCake
http://purl.org/heals/ingredient/WholeGrainBananaPancake
http://purl.org/heals/ingredient/SaucyShepherdPi

## HI ontology exploration

In your project, you will be working with a Hybrid Intelligence (HI) ontology. This is an opportunity for you to get acquainted with its structure. Applying the skills from the exercises above perform the following actions:

1. Load the HI ontology from the data folder (hi_ontology.ttl) with RDFlib.
2. Create an "HI" Namespace.
3. Count the number of triples.
4. List all subjects.
5. List all predicates.
6. List all pairs of subjects and their corresponding objects linked by a rdf:type predicate.

In [18]:
# Load the HI ontology and create a HI namespace
hi_graph = Graph()
hi_graph.parse("data/hi_ontology.ttl")

HI = Namespace("http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/>")

In [19]:
# Count the number of triples
print("Triples in the HI_graph:", len(hi_graph))

Triples in the HI_graph: 114


In [20]:
# List all the subjects
print(f'\nPrinting all subjects')
for sub in hi_graph:
    print(sub)

print("Total number of subjects:", sum(1 for _ in hi_graph.triples((None, RDF.type, None))))


Printing all subjects
(rdflib.term.URIRef('http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Adaptiveness'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#NamedIndividual'))
(rdflib.term.URIRef('http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/interactionTask'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#ObjectProperty'))
(rdflib.term.URIRef('http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Adaptiveness'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Capability'))
(rdflib.term.URIRef('http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/interactionMethod'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#range'), rdflib

In [21]:
# List all predicates
print(f'\nPrinting all predicates')
for pred in hi_graph:
    print(pred)

print("Total number of predicates:", sum(1 for _ in hi_graph.triples((None, RDF.type, None))))


Printing all predicates
(rdflib.term.URIRef('http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Adaptiveness'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#NamedIndividual'))
(rdflib.term.URIRef('http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/interactionTask'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#ObjectProperty'))
(rdflib.term.URIRef('http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Adaptiveness'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Capability'))
(rdflib.term.URIRef('http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/interactionMethod'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#range'), rdfl

In [22]:
# List all pairs of subjects and their corresponding objects linked by a rdf:type predicate
for sub, pred, obj in hi_graph.triples((None, RDF.type, None)):
    print(f'Subject: {sub}, --- Object: {obj}')

print("Total number of pairs:", sum(1 for _ in hi_graph.triples((None, RDF.type, None))))

Subject: http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51, --- Object: http://www.w3.org/2002/07/owl#Ontology
Subject: http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Adaptiveness, --- Object: http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Capability
Subject: http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/BDI, --- Object: http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Capability
Subject: http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Collaborativeness, --- Object: http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Capability
Subject: http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Communication, --- Object: http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Capability
Subject: http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Responsibility, ---